# 02 — QSAR on PyTDC tasks with DeepChem, score the MCAS library

> ⚠️ **Not medical advice.** Research/hypothesis use only.

What this notebook does:
1. Loads a PyTDC ADMET / Tox task (default: hERG cardiac liability — relevant for any mast cell stabilizer).
2. Trains a DeepChem graph-conv model on the task.
3. Scores every small molecule in `MCAS_Compound_Library_v1.csv` and writes ranked predictions to `outputs/qsar_predictions.csv`.

CPU-feasible on Mac (10–20 min per task). For larger sweeps use Colab GPU.

## Install

```bash
pip install -e .[qsar]   # from repo root
```
This pulls `deepchem`, `scikit-learn`, and `torch` on top of the base deps.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
LIBRARY_CSV = REPO_ROOT / 'data' / 'compounds' / 'MCAS_Compound_Library_v1.csv'
OUT_DIR = REPO_ROOT / 'outputs'
OUT_DIR.mkdir(exist_ok=True)
library = pd.read_csv(LIBRARY_CSV)
smiles_lib = library[library['canonical_smiles'].notna() & (library['canonical_smiles'] != '')]
len(smiles_lib)

In [ ]:
from tdc.single_pred import Tox
TASK_NAME = 'hERG'  # try also: 'AMES', 'DILI', 'Carcinogens_Lagunin'
task = Tox(name=TASK_NAME)
split = task.get_split()
train_df, valid_df, test_df = split['train'], split['valid'], split['test']
print('train', len(train_df), 'valid', len(valid_df), 'test', len(test_df))
train_df.head()

In [ ]:
import deepchem as dc
featurizer = dc.feat.ConvMolFeaturizer()

def to_dataset(df):
    feats = featurizer.featurize(df['Drug'].tolist())
    mask = np.array([f is not None and not (isinstance(f, np.ndarray) and f.size == 0) for f in feats])
    feats = np.array(feats, dtype=object)[mask]
    y = np.array(df['Y'].tolist())[mask]
    return dc.data.NumpyDataset(X=feats, y=y, ids=np.array(df['Drug'].tolist())[mask])

train_ds = to_dataset(train_df)
valid_ds = to_dataset(valid_df)
test_ds = to_dataset(test_df)
len(train_ds), len(valid_ds), len(test_ds)

In [ ]:
model = dc.models.GraphConvModel(
    n_tasks=1,
    mode='classification',
    graph_conv_layers=[64, 64],
    dense_layer_size=128,
    dropout=0.2,
    learning_rate=1e-3,
    batch_size=64,
)
model.fit(train_ds, nb_epoch=20)
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
print('valid AUC:', model.evaluate(valid_ds, [metric])['roc_auc_score'])
print('test  AUC:', model.evaluate(test_ds, [metric])['roc_auc_score'])

In [ ]:
lib_feats = featurizer.featurize(smiles_lib['canonical_smiles'].tolist())
lib_ds = dc.data.NumpyDataset(X=lib_feats, ids=smiles_lib['name'].tolist())
preds = model.predict(lib_ds)
scores = preds[:, 0, 1] if preds.ndim == 3 else preds[:, 0]
result = smiles_lib[['name', 'category', 'subcategory', 'evidence_level']].copy()
result[f'{TASK_NAME}_score'] = scores
out_path = OUT_DIR / 'qsar_predictions.csv'
result.sort_values(f'{TASK_NAME}_score').to_csv(out_path, index=False)
print('wrote', out_path)
result.sort_values(f'{TASK_NAME}_score').head(15)

## Interpretation

Lower hERG score = lower predicted cardiac liability — a key filter for mast cell stabilizers because many first-gen antihistamines and KIT inhibitors have hERG issues.

Repeat with `TASK_NAME = 'AMES'` (mutagenicity), `'DILI'` (liver), `'BBB_Martins'` (BBB penetration — relevant for neuro-MCAS hypotheses). Then `05_hypothesis_ranking.ipynb` joins all these scores with docking and evidence to rank candidates per category.